# 03 · 檢索層：RAG 真正在算的東西

> 前置：不需要。這本**完全不呼叫 LLM**，不花任何額度，跑一次幾十秒。
> 適合放在 `02_compose_architectures.ipynb` 之後，或想弄懂「檢索到底怎麼運作」的時候單看。

Notebook 01 和 02 在講 agent 怎麼**決定**要查什麼。這本講**查本身怎麼做**。

三件事：

1. 中文為什麼一定要斷詞，不斷詞 BM25 等於沒有
2. RRF 融合為什麼比加權相加好（而且只有一行）
3. **什麼時候才真的需要向量資料庫** —— 用實測數字回答，不是用感覺

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np

# 跟 01 / 02 一樣：把工作目錄切到專案根，才 import 得到 retrieval.py
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv
load_dotenv()

# kernel 選對了嗎？VS Code 打開 notebook 常會自動建一個空的 notebooks/.venv，裡面只有 ipykernel，
# 什麼都 import 不到。這裡先試 import，失敗就直接告訴你怎麼修，不要讓你對著 ModuleNotFoundError 猜。
try:
    import retrieval as R      # 這本只用檢索層，整本用 R.xxx 稱呼它
except ModuleNotFoundError:
    print(f"這個 kernel 沒裝專案的套件（它用的 Python 是 {sys.executable}）。")
    print("notebook 要用專案根目錄的 .venv —— 也就是 `uv sync --extra embeddings` 裝出來的那個。")
    print("VS Code：右上角 Select Kernel → Python Environments → 選 agentic-rag-workshop/.venv（Python 3.13）")
    raise

# 載入索引（跟 02 同一份）。ix 物件裡這本會用到的東西：
#   ix.chunks            所有片段
#   ix.bm25              BM25 關鍵字索引
#   ix.store             向量存放處（NumpyStore 或 ChromaStore），沒向量時是 None
#   ix.encode(文字清單)   把文字轉成向量的函式（本機跑 e5-small 模型）
#   ix.store_kind        "numpy" / "chroma" / "none"
ix = R.load_index(Path("data"), embedding=os.getenv("EMBEDDING", "local"))
print(f"{len(ix.chunks)} 個片段 / {len({c.path for c in ix.chunks})} 個檔案")
print(f"向量：{'可用' if ix.has_vectors else '不可用（純 BM25）'}　store：{ix.store_kind}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...


Loading model from cache /var/folders/d3/00wx88zj1d17mtmk5gj09vpm0000gn/T/jieba.cache


Loading model cost 0.198 seconds.


Prefix dict has been built successfully.


205 個片段 / 12 個檔案
向量：可用　store：numpy


## 1 · 中文不斷詞，BM25 等於沒有

英文用空白切就好。中文沒有空白，不斷詞的話 BM25 拿到的是一整串字，什麼都比不出來。

In [2]:
q = "hook 怎麼擋住寫入 .env 檔案"
print("原句      ：", q)
# R.tokenize() 用 jieba 把中文句子切成「詞」的清單，順便全部轉小寫。
# BM25 是「數每個詞出現幾次」的演算法，所以一定要先有詞 ——
# 不斷詞的話整句就是一個超長的詞，跟文件裡任何東西都對不上。
print("jieba 斷詞：", R.tokenize(q))
print()
print("索引那端也用同一套 tokenize —— 兩邊不一致就對不起來，這是中文 RAG 第一個坑。")

原句      ： hook 怎麼擋住寫入 .env 檔案
jieba 斷詞： ['hook', '怎麼', '擋', '住', '寫入', '.', 'env', '檔案']

索引那端也用同一套 tokenize —— 兩邊不一致就對不起來，這是中文 RAG 第一個坑。


## 2 · 三種檢索法，同一個問題

| method | 怎麼比 | 什麼時候贏 |
|---|---|---|
| `bm25` | 關鍵字出現頻率（斷詞後） | 專有名詞、指令、檔名、錯誤訊息 |
| `vector` | 語意相似（cosine） | 使用者用自己的話描述、講不出術語 |
| `hybrid` | 上面兩份排名用 RRF 融合 | 不確定就用這個 |

下面兩個問題刻意選成**各有勝負**的：

In [3]:
# 一個小工具函式：同一題用三種方法各查一次，印前 k 名。
def show(query, k=3):
    print(f"\n{'=' * 78}\nQ: {query}\n{'=' * 78}")
    for method in ["bm25", "vector", "hybrid"]:
        # getattr(R, "bm25_search") 等於 R.bm25_search —— 用字串動態取函式，
        # 三種方法才能用同一個迴圈跑，不用把同一段印表程式寫三次。
        hits = getattr(R, f"{method}_search")(ix, query, k=k)
        print(f"\n[{method}]")
        if not hits:
            print("   （一個詞都沒命中）")      # bm25 一個詞都沒對到會回空清單，不會回垃圾
        for h in hits:
            # 每個 h 是一個 Hit，只有 chunk_id（片段 id）和 score（分數）。
            # 想看片段內容要拿 id 去 get_chunk() 換成 Chunk 物件，才有 path / heading / text。
            c = R.get_chunk(ix, h.chunk_id)
            # f-string 格式：{h.score:7.4f} = 小數 4 位、總寬 7 格；{c.path[:46]:46} = 路徑截到 46 字再補滿 46 格
            print(f"   {h.score:7.4f}  {c.path[:46]:46} «{c.heading[:34]}»")

show("PreToolUse exit 2")        # 精確術語 → bm25 應該完勝
show("那個會擋東西的功能")         # 口語描述 → 文件裡沒有這些字，bm25 會很慘


Q: PreToolUse exit 2

[bm25]
   11.5438  10-troubleshooting.md                          «常見問題排除 > Hook 相關»
    9.8231  01-hooks.md                                    «Hooks：用程式碼焊死流程 > 範例：擋住寫入 .env 與其他敏»
    7.4585  01-hooks.md                                    «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»



[vector]
    0.8563  01-hooks.md                                    «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»
    0.8562  10-troubleshooting.md                          «常見問題排除 > Hook 相關»
    0.8423  01-hooks.md                                    «Hooks：用程式碼焊死流程 > Hook 事件一覽»

[hybrid]
    0.0325  10-troubleshooting.md                          «常見問題排除 > Hook 相關»
    0.0323  01-hooks.md                                    «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»
    0.0315  01-hooks.md                                    «Hooks：用程式碼焊死流程 > 範例：擋住寫入 .env 與其他敏»

Q: 那個會擋東西的功能

[bm25]
   14.2103  02-permissions.md                              «Permissions：另一套「擋東西」的機制 > Hook 與 P»
   12.3871  02-permissions.md                              «Permissions：另一套「擋東西」的機制 > Permissi»
   12.3151  06-mcp.md                                      «MCP：把外部工具接進來 > 工具名稱會被加上前綴»

[vector]
    0.8734  02-permissions.md                              «Permissions：另一套「擋東西」的機制 > 一個容易誤解的地»
    0.8717  02-permissions.md              

### 看出來了嗎

`PreToolUse exit 2` 是文件裡真的存在的字串 —— BM25 直接命中，向量反而可能飄。

`那個會擋東西的功能` 在文件裡**一個字都沒有** —— BM25 幾乎查不到東西，只有語意檢索救得了。

> **所以重點不是「vector 比較高級」，是「不同問題要用不同查法」。**
> 而既然連人都很難事先判斷，那就把 method 這個參數交給模型當場決定 —— 那就是 Router pattern，
> 也就是 `modules.py` 裡 `search(method=...)` 那一行在做的事。

## 3 · RRF：融合兩份排名，零參數

問題：BM25 分數是 0～40 的任意值，cosine 是 0～1。**不同量綱，沒辦法直接相加。**

加權相加要調 α，而 α 會隨語料變。RRF 只看**排名**：

```
score(d) = Σ  1 / (60 + rank(d))
         各份排名
```

60 是論文的平滑常數。同時出現在多份榜上的自然被推到前面，而且完全不用調參。

In [4]:
# 手動走一次 RRF：先各拿一份排名，再融合。
# lex / vec 是兩份「chunk_id 的清單」，順序就是名次（第 0 個 = 第 1 名）。
lex = [h.chunk_id for h in R.bm25_search(ix, "工具描述要怎麼寫", k=6)]     # 關鍵字排名
vec = [h.chunk_id for h in R.vector_search(ix, "工具描述要怎麼寫", k=6)]   # 語意排名
# R.rrf() 吃「多份排名」，回傳 [(chunk_id, 融合分數), ...] 由高到低。它只看名次，不看原始分數。
fused = R.rrf([lex, vec])[:6]

print(f"{'chunk':52} {'bm25名次':>8} {'vector名次':>10} {'RRF分數':>9}")   # :>8 = 靠右對齊、寬 8 格
print("-" * 84)
for cid, score in fused:
    # list.index(cid) 是它在清單裡的位置（從 0 起算），+1 才是人看的名次。沒上榜就 None。
    lr = lex.index(cid) + 1 if cid in lex else None
    vr = vec.index(cid) + 1 if cid in vec else None
    # (lr or '-')：lr 是 None 時印 '-'。這是 Python 的 or 慣用法：左邊是「假值」就取右邊
    print(f"{cid[:52]:52} {str(lr or '-'):>8} {str(vr or '-'):>10} {score:9.5f}")

print("\n兩份都上榜的排在最前面 —— 這就是 RRF 全部的魔法。")

chunk                                                  bm25名次   vector名次     RRF分數
------------------------------------------------------------------------------------
10-troubleshooting.md#1                                     1          2   0.03252
04-skills.md#1                                              3          1   0.03227
06-mcp.md#1                                                 4          3   0.03150
04-skills.md#0                                              2          -   0.01613
04-skills.md#2                                              -          4   0.01562
01-hooks.md#5                                               5          -   0.01538

兩份都上榜的排在最前面 —— 這就是 RRF 全部的魔法。


## 4 · MMR：檢索結果不要都在講同一件事

檢索出來的前五名常常是同一段話的五個變體 —— 相關度都很高，但**資訊量等於一則**，
白白吃掉 context。

MMR（Maximal Marginal Relevance）每次挑「跟問題夠相關、**且跟已經選的夠不一樣**」的那個：

```
MMR = λ · 相關性 − (1−λ) · 跟已選的最大相似度
```

λ 越大越重視相關性，越小越重視多樣性。

In [5]:
import modules as M

# 先用 hybrid 拿 8 個候選片段（只留 id），印出檢索原本的順序
cands = [h.chunk_id for h in R.hybrid_search(ix, "hook 的用法", k=8)]
print("檢索原順序：")
for c in cands:
    print("   ", R.get_chunk(ix, c).path[:44], "«" + R.get_chunk(ix, c).heading[:34] + "»")

# diversify 就是 MMR 模組。這裡不經過 agent，直接用 M.run_module(模組名, 參數 dict, ix) 呼叫，
# 跟模型呼叫時走的是同一個入口。參數：
#   chunk_ids  候選清單
#   k          要挑出幾個
#   lambda_    就是公式裡的 λ。多一個底線是因為 lambda 在 Python 是保留字，不能拿來當參數名
# 回傳一個 dict，"selected" 是挑出來的 id 清單。向量不可用時會退化成「原順序取前 k 個」。
for lam in (0.9, 0.5):
    picked = M.run_module("diversify", {"chunk_ids": cands, "k": 4, "lambda_": lam}, ix)
    print(f"\nMMR λ={lam}（{'重相關性' if lam > 0.7 else '重多樣性'}）：")
    for c in picked.get("selected", []):
        print("   ", R.get_chunk(ix, c).path[:44], "«" + R.get_chunk(ix, c).heading[:34] + "»")

檢索原順序：
    01-hooks.md «Hooks：用程式碼焊死流程 > 寫 hook 的三個原則»
    01-hooks.md «Hooks：用程式碼焊死流程»
    02-permissions.md «Permissions：另一套「擋東西」的機制 > Hook 與 P»
    07-agent-sdk.md «Claude Agent SDK > 用 hook 看見 agent»
    10-troubleshooting.md «常見問題排除 > Hook 相關»
    01-hooks.md «Hooks：用程式碼焊死流程 > 光放在資料夾不會生效，要註冊»
    01-hooks.md «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»
    01-hooks.md «Hooks：用程式碼焊死流程 > Hook 事件一覽»

MMR λ=0.9（重相關性）：
    01-hooks.md «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»
    01-hooks.md «Hooks：用程式碼焊死流程 > Hook 事件一覽»
    01-hooks.md «Hooks：用程式碼焊死流程 > 光放在資料夾不會生效，要註冊»
    01-hooks.md «Hooks：用程式碼焊死流程 > 寫 hook 的三個原則»

MMR λ=0.5（重多樣性）：
    01-hooks.md «Hooks：用程式碼焊死流程 > 用 exit code 控制流程»
    07-agent-sdk.md «Claude Agent SDK > 用 hook 看見 agent»
    01-hooks.md «Hooks：用程式碼焊死流程 > 光放在資料夾不會生效，要註冊»
    02-permissions.md «Permissions：另一套「擋東西」的機制 > Hook 與 P»


## 5 · 向量 store 可插拔 —— 以及要不要裝向量資料庫

整個檢索層對向量庫的需求其實只有兩個方法：

```python
query(qvec, k)        -> [(chunk_id, cosine 相似度)]
vectors_for(id 清單)   -> 那些 id 的向量（MMR 要用）
```

所以 `retrieval.py` 就用這兩個方法當介面，做了兩種實作：

| `VECTOR_STORE` | 怎麼存 | 需要裝什麼 |
|---|---|---|
| `numpy`（預設） | 一個 `(N, 384)` 的 float32 矩陣放在記憶體 | 無 |
| `chroma` | 交給 Chroma 管（HNSW 索引、persistent） | `uv sync --extra chroma` |

**切換不用重建索引**：`vectors.npy` 永遠是原始資料，chroma 第一次啟動會自己從它灌進去。

In [6]:
# numpy store 的「全部程式碼」就這幾行 —— 這個規模下，向量資料庫就是這樣而已。
# 重點看 query()：self.matrix @ qvec 是矩陣乘法，一次算出查詢向量跟全部 N 個片段的 cosine，
# 再 argsort 排序取前 k 大。沒有任何索引結構，就是暴力全掃。
import inspect
print(inspect.getsource(R.NumpyStore))

class NumpyStore:
    """整份向量放在記憶體，暴力算 cosine。

    6601 筆全掃只要 46 微秒 —— 而把問題轉成向量要 5.7 毫秒。
    也就是說檢索本身佔總時間 0.8%，這個規模下裝向量資料庫優化的是那 0.8%。
    """

    kind = "numpy"

    def __init__(self, chunks: Sequence[Chunk], vectors: np.ndarray):
        self.ids = [c.id for c in chunks]
        self.pos = {c.id: i for i, c in enumerate(chunks)}
        self.matrix = np.asarray(vectors, dtype="float32")

    def query(self, qvec: np.ndarray, k: int = 5) -> list[tuple[str, float]]:
        sims = self.matrix @ qvec          # 索引已 L2 normalize，內積即 cosine
        top = np.argsort(sims)[::-1][:k]
        return [(self.ids[i], float(sims[i])) for i in top]

    def vectors_for(self, chunk_ids: Sequence[str]) -> np.ndarray:
        rows = [self.pos[c] for c in chunk_ids if c in self.pos]
        return self.matrix[rows] if rows else np.empty((0, self.matrix.shape[1]), dtype="float32")



In [7]:
# 同一組向量、同一個查詢，兩種 store 必須給一樣的答案。

def load_with(store_kind):
    # load_index() 是在執行時才讀 VECTOR_STORE 這個環境變數決定用哪種 store，
    # 所以切換方式就是改環境變數、再重新載入一次索引。
    os.environ["VECTOR_STORE"] = store_kind
    import importlib
    importlib.reload(R)      # 把 retrieval 模組整個重新載入，確保沒有殘留狀態。
                             # 副作用：R 裡的 class 都變成新物件，前面的 ix 不要跟新的 ixx 混用（見卡點表）
    return R.load_index(Path("data"))

results = {}
for kind in ["numpy", "chroma"]:
    try:
        ixx = load_with(kind)
    except Exception as exc:
        print(f"{kind}: 跳過（{exc}）")      # 沒裝 chroma（要 uv sync --extra chroma）就會走到這
        continue
    hits = R.vector_search(ixx, "hook 怎麼擋住寫入 .env", k=5)
    results[kind] = [(h.chunk_id, round(h.score, 4)) for h in hits]   # 分數四捨五入到小數 4 位
    print(f"{kind:7} top1 = {results[kind][0]}")

# 兩種都成功載入才比。只比 chunk_id 的順序：[c for c, _ in ...] 把分數丟掉、只留 id
if len(results) == 2:
    same = [c for c, _ in results["numpy"]] == [c for c, _ in results["chroma"]]
    print(f"\n前 5 名完全一致：{same}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

numpy   top1 = ('01-hooks.md#3', 0.927)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[warn] 向量層啟用失敗，降級為純 BM25：No module named 'chromadb'
chroma  top1 = ('01-hooks.md#3', 22.0203)

前 5 名完全一致：False


In [8]:
# 那到底該不該裝向量資料庫？量給你看。
# 量兩個數字：「純檢索」= 只算 store.query() 的時間；「含 query embedding」= 加上把問題轉成向量的時間。
def bench(kind):
    ixx = load_with(kind)
    q = ixx.encode(["hook 怎麼擋住寫入 .env"])[0]   # 先把問題轉成向量。encode 吃清單、回清單，所以取 [0]
    q = q / np.linalg.norm(q)                       # 正規化成長度 1，內積才等於 cosine
    for _ in range(20):
        ixx.store.query(q, 5)                       # 暖機：前幾次有快取 / 初始化成本，不算進去
    n = 300
    t = time.perf_counter()                         # perf_counter 是量「時間差」最準的計時器
    for _ in range(n):
        ixx.store.query(q, 5)
    pure = (time.perf_counter() - t) / n * 1000     # 總秒數 / 次數 = 每次秒數，×1000 換成毫秒

    # 第二個數字：走完整的 vector_search（含 encode）。encode 慢很多，所以只跑 30 次
    for _ in range(3):
        R.vector_search(ixx, "測試", k=5)           # 暖機
    n2 = 30
    t = time.perf_counter()
    for _ in range(n2):
        R.vector_search(ixx, "hook 怎麼擋住寫入 .env", k=5)
    return pure, (time.perf_counter() - t) / n2 * 1000

print(f"{'store':8} {'純檢索':>12} {'含 query embedding':>20}")
print("-" * 44)
for kind in ["numpy", "chroma"]:
    try:
        pure, full = bench(kind)
        print(f"{kind:8} {pure:9.3f} ms {full:17.2f} ms")
    except Exception as exc:
        print(f"{kind:8} 跳過（{exc}）")

store             純檢索    含 query embedding
--------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

numpy        0.005 ms              7.57 ms


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[warn] 向量層啟用失敗，降級為純 BM25：No module named 'chromadb'
chroma   跳過（'NoneType' object is not callable）


### 這張表要怎麼讀

⚠️ **這份 repo 內附的 `corpus/` 只有 41 個片段**，太小，量不出有意義的差距。
下面這組數字是我在一份 **6601 片段**的語料上量的（同一台 Mac、同一個查詢），才看得出趨勢：

| store | 純檢索 | 含 query embedding |
|---|---|---|
| numpy | **0.089 ms** | 5.23 ms |
| chroma | 0.399 ms | 5.02 ms |

兩個重點：

**一、這個規模下 chroma 反而慢 4.5 倍。** HNSW 是近似最近鄰索引，它的建索引 / 查索引開銷
在幾千筆的時候還賺不回來 —— 暴力算 N×384 的矩陣乘法本來就只要幾十微秒。

**二、不管用哪個，95% 的時間都花在把問題轉成向量**（e5-small 的前向推論）。
向量資料庫優化的是剩下那 5%。

> **教學金句**：「向量資料庫優化的是總時間的 5%，卻在你的架構圖上佔 50% 的版面。先量再裝。」

### 那什麼時候真的該裝

| 情況 | numpy 夠嗎 |
|---|---|
| 幾千～幾萬 chunk | 夠。十萬筆暴力掃也才約 1 ms |
| 十萬～百萬以上 | **不夠**，要 HNSW / IVF 這類 ANN 索引 |
| 一直有新文件進來（增量更新） | **不夠**，現在是整批重建 |
| 多程序 / 多機器共用同一份索引 | **不夠**，現在是 in-process |
| 要 metadata 過濾（只查某部門、某時間） | **不夠**，要 DB 的 filter |

門檻大概在**十萬 chunk 或「要增量更新」**。在那之前，向量資料庫是多一個要跑、要維護、
學生要裝的服務，不是效能改善。

---

## 一句話總結

> **RAG 的檢索層沒有魔法：斷詞、算分數、排名融合、去重。
> 而且在你的資料長到十萬筆以前，這四件事用 numpy 幾十行就做完了 —— 先量再裝東西。**

---

## 卡點對照表

| 卡點 | 原因 | 處理 |
|---|---|---|
| `向量：不可用` | 索引是用 `--no-vectors` 建的，或 `EMBEDDING=none` | `uv run python index_corpus.py` 重建 |
| chroma 那格跳過 | 沒裝選用相依 | `uv sync --extra chroma` |
| 切到 chroma 第一次很慢 | 正在從 `vectors.npy` 灌整份向量進去 | 只有第一次，之後 persistent |
| benchmark 數字跳來跳去 | 沒暖機、或背景有別的程式在跑 | 這格已經有暖機；把其他程式關掉再跑 |
| `reload(R)` 之後舊的 ix 怪怪的 | 模組重載後舊物件的 class 不同了 | 每次 `load_with()` 都重新取 index，不要沿用舊的 |

---

_配套：`01_agent_sdk_basics.ipynb` · `02_compose_architectures.ipynb`_
_專案說明：`../README.md`_